In [1]:
!ollama pull deepseek-v2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling d8d69f2a1bfa: 100% ▕██████████████████▏ 8.9 GB                         
pulling b321cd7de6c7: 100% ▕██████████████████▏  111 B                         
pulling 4bb71764481f: 100% ▕██████████████████▏  13 KB                         
pulling 19f2fb9e8bc6: 100% ▕██████████████████▏   32 B                         
pulling 2b2c35718a16: 100% ▕██████████████████▏  494 B                         
verifying sha256 digest 
writing manifest 
removing unused layers ⠋ pulling manifest 
pulling d8d69f2a1bfa: 100% ▕██████████████████▏ 8.9 GB                         
pulling b321cd7de6c7: 100% ▕██████████████████▏  111 B                         
pulling 4bb71764481f: 100% ▕██████████████████▏  13

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

# If you are not using a .env file, uncomment and fill these:
# os.environ["GROQ_API_KEY"] = "your-groq-api-key"
# os.environ["GOOGLE_API_KEY"] = "your-google-api-key"

GROQ_MODEL = os.getenv("GROQ_API_KEY", "qwen/qwen3.8-27b")
GOOGLE_MODEL = os.getenv("GOOGLE_API_KEY", "gemini-3.1-flash-lite")
OLLAMA_MODEL = os.getenv("OLLAMA_API_KEY", "deepseek-v2")

In [26]:
import requests
import gradio as gr

from langchain.agents import create_agent
from langchain.tools import tool

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

In [27]:
@tool
def get_weather(city: str) -> dict:
    """Get the current weather and a 3-day forecast for a city worldwide."""

    try:
        geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_response = requests.get(
            geocoding_url,
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json",
            },
            timeout=20,
        )
        geo_response.raise_for_status()
        geo_results = geo_response.json().get("results", [])

        if not geo_results:
            return {"error": f"Could not find a location matching '{city}'."}

        place = geo_results[0]

        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": [
                    "temperature_2m",
                    "relative_humidity_2m",
                    "apparent_temperature",
                    "precipitation",
                    "wind_speed_10m",
                    "weather_code",
                ],
                "daily": [
                    "temperature_2m_max",
                    "temperature_2m_min",
                    "precipitation_probability_max",
                ],
                "forecast_days": 3,
                "timezone": "auto",
            },
            timeout=20,
        )
        weather_response.raise_for_status()
        weather_data = weather_response.json()

        return {
            "location": {
                "name": place.get("name"),
                "admin1": place.get("admin1"),
                "country": place.get("country"),
            },
            "timezone": weather_data.get("timezone"),
            "current_weather": weather_data.get("current"),
            "three_day_forecast": weather_data.get("daily"),
            "units": {
                "temperature": "°C",
                "wind_speed": "km/h",
                "precipitation": "mm",
            },
        }

    except requests.RequestException as exc:
        return {"error": f"Weather service request failed: {exc}"}

In [28]:
def get_model(provider: str):
    if provider == "Groq":
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY is missing. Add it to .env or os.environ.")
        return ChatGroq(
            model="qwen/qwen3.8-27b",
            temperature=0,
            api_key=api_key,
        )

    if provider == "Ollama":
        return ChatOllama(
            model="deepseek-v2",
            temperature=0,
        )

    if provider == "Google":
        api_key = os.getenv("GOOGLE_API_KEY")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY is missing. Add it to .env or os.environ.")
        return ChatGoogleGenerativeAI(
            model="gemini-3.1-flash-lite",
            temperature=0,
            google_api_key=api_key,
        )

    raise ValueError(f"Unsupported provider: {provider}")

In [29]:
SYSTEM_PROMPT = """
You are a helpful AI Weather Assistant.

Rules:
- Use the get_weather tool for requests about current weather or forecasts.
- If the user asks about multiple cities, call the tool for each city.
- Never invent weather observations or forecast values.
- Explain temperatures in Celsius and wind speed in km/h.
- Clearly distinguish current conditions from forecast information.
- If the city is unclear or missing, ask the user to specify it.
- Present results in readable Markdown.
"""

def create_weather_agent(provider: str):
    return create_agent(
        model=get_model(provider),
        tools=[get_weather],
        system_prompt=SYSTEM_PROMPT,
    )

In [30]:
def extract_text(content) -> str:
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
        return "\\n".join(part for part in parts if part)

    return str(content)

In [31]:
def chat(message, history, provider):
    if not message or not message.strip():
        return "", history

    try:
        agent = create_weather_agent(provider)

        conversation = []
        for item in (history or []):
            role = item.get("role")
            content = item.get("content")

            # Only pass plain text user/assistant messages to the agent.
            if role in ("user", "assistant") and isinstance(content, str):
                conversation.append({
                    "role": role,
                    "content": content,
                })

        conversation.append({
            "role": "user",
            "content": message.strip(),
        })

        result = agent.invoke({"messages": conversation})
        answer = extract_text(result["messages"][-1].content)

        updated_history = (history or []) + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer},
        ]
        return "", updated_history

    except Exception as exc:
        error_history = (history or []) + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": f"Error: {type(exc).__name__}: {exc}"},
        ]
        return "", error_history

In [32]:
with gr.Blocks(theme=gr.themes.Soft(), title="Multi-Model AI Weather Agent") as demo:
    gr.Markdown(
        """
        # 🌦️ Multi-Model AI Weather Agent
        Ask about current weather or a 3-day forecast.
        Powered by **LangChain**, **Open-Meteo**, and your selected LLM.
        """
    )

    provider = gr.Dropdown(
        choices=["Groq", "Ollama", "Google"],
        value="Groq",
        label="Choose AI model",
    )

    chatbot = gr.Chatbot(
        height=480,
        label="Weather Assistant",
    )

    with gr.Row():
        message = gr.Textbox(
            placeholder="Example: What is the current weather in Pune?",
            label="Your message",
            scale=8,
        )
        send = gr.Button("Send", variant="primary", scale=1)

    clear = gr.ClearButton(
        components=[message, chatbot],
        value="Clear chat",
    )

    send.click(
        fn=chat,
        inputs=[message, chatbot, provider],
        outputs=[message, chatbot],
    )
    message.submit(
        fn=chat,
        inputs=[message, chatbot, provider],
        outputs=[message, chatbot],
    )

C:\Users\Hrishikesh Kulkarni\AppData\Local\Temp\ipykernel_15964\1400423916.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Multi-Model AI Weather Agent") as demo:


In [11]:
import gradio as gr

chatbot = gr.Chatbot(
    
    height=480,
    label="Weather Assistant"
)

In [33]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
